[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/anderson-perez/IAC/blob/main/4%20-%20Aprendizado%20Profundo/Fundamentos_de_YOLO.ipynb)

# Inteligência Artificial e Computacional

## Segunda Parte da Disciplina

# Informações:

- Prof. Anderson Luiz Fernandes Perez
- Contato: anderson.perez@ufsc.br | anderson.lfp@gmail.com
- Os materiais da disciplina estarão disponíveis no github (https://github.com/anderson-perez/IAC.git)

# Fundamentos de YOLO (*You Only Look Once*)


A ideia original do YOLO é tratar a detecção de objetos como um único problema de regressão: uma rede neural recebe a imagem inteira e, em uma única passada (*forward pass*), prediz caixas delimitadoras e probabilidades de classes.

O artigo original é:

> Redmon, J.; Divvala, S.; Girshick, R.; Farhadi, A. **You Only Look Once: Unified, Real-Time Object Detection**. CVPR, 2016.  
> https://arxiv.org/abs/1506.02640

## Detecção de Objetos

A **detecção de objetos** é uma tarefa de visão computacional em que o modelo precisa responder simultaneamente:

1. **Onde está o objeto?** Por meio de uma caixa delimitadora (*bounding box*).
2. **Que objeto é esse?** Por meio de uma classe, como `pessoa`, `carro`, `cachorro`, `semáforo` etc.

Diferentemente da **classificação de imagens**, em que o modelo produz apenas um rótulo para a imagem inteira, a detecção de objetos localiza múltiplos objetos na mesma imagem.

```text
Imagem → Modelo detector → [(classe='carro', caixa=[x1,y1,x2,y2], confiança=0.91),
                            (classe='pessoa', caixa=[x1,y1,x2,y2], confiança=0.87)]
```

A principal inovação do YOLO foi unificar o pipeline de detecção, permitindo detecção em tempo real.


## Detectores de dois estágios versus detectores de um estágio

Detectores de **dois estágios**, como R-CNN, Fast R-CNN e Faster R-CNN, geralmente primeiro geram regiões candidatas e depois classificam/refinam essas regiões.

O YOLO pertence à família dos detectores de **um estágio** (*one-stage detectors*):

1. A imagem entra na rede.
2. A rede prediz diretamente caixas, classes e confianças.

Isso tende a tornar o processo mais rápido, embora exista um compromisso entre velocidade e precisão.


## Representação de uma Caixa Delimitadora

Uma caixa delimitadora pode ser representada de diferentes formas.

### Coordenadas dos cantos

$$
(x_1, y_1, x_2, y_2)
$$

- $(x_1, y_1)$: canto superior esquerdo;
- $(x_2, y_2)$: canto inferior direito.

### Centro, largura e altura

$$
(x_c, y_c, w, h)
$$

- $(x_c, y_c)$: centro da caixa;
- $(w)$: largura;
- $(h)$: altura.


In [ ]:
# Exemplo simples de conversão entre formatos de bounding box

def xywh_to_xyxy(xc, yc, w, h):
    x1 = xc - w / 2
    y1 = yc - h / 2
    x2 = xc + w / 2
    y2 = yc + h / 2
    return x1, y1, x2, y2

box_xywh = (100, 80, 40, 30)
print("Formato (xc, yc, w, h):", box_xywh)
print("Formato (x1, y1, x2, y2):", xywh_to_xyxy(*box_xywh))


## Interseção sobre União: IoU

A métrica **IoU** (*Intersection over Union*) mede a sobreposição entre duas caixas.


IoU = {área da interseção}{área da união}


- IoU = 0: não há sobreposição;
- IoU próximo de 1: caixas muito parecidas;
- IoU ≥ 0.5: frequentemente usado como critério mínimo em avaliações básicas.


In [ ]:
def iou(boxA, boxB):
    # Calcula IoU entre duas caixas no formato (x1, y1, x2, y2).
    xA = max(boxA[0], boxB[0])
    yA = max(boxA[1], boxB[1])
    xB = min(boxA[2], boxB[2])
    yB = min(boxA[3], boxB[3])

    inter_w = max(0, xB - xA)
    inter_h = max(0, yB - yA)
    inter_area = inter_w * inter_h

    areaA = max(0, boxA[2] - boxA[0]) * max(0, boxA[3] - boxA[1])
    areaB = max(0, boxB[2] - boxB[0]) * max(0, boxB[3] - boxB[1])

    union_area = areaA + areaB - inter_area
    return 0 if union_area == 0 else inter_area / union_area

box_real = (50, 50, 150, 150)
box_pred = (70, 70, 170, 170)
print("IoU:", iou(box_real, box_pred))


## Confiança da Predição

Em detectores de objetos, cada predição costuma ter um valor de **confiança**.

De forma intuitiva, a confiança combina duas ideias:

1. A probabilidade de existir um objeto naquela região.
2. A qualidade da localização da caixa.

Em versões clássicas do YOLO, a confiança era frequentemente interpretada como:

$$
\text{Confiança} = P(\text{Objeto}) \times IoU
$$

Em implementações modernas, os detalhes mudam, mas a intuição permanece: uma boa detecção deve ter alta probabilidade de classe e boa localização.


## Non-Maximum Suppression (NMS)

Um detector pode prever várias caixas para o mesmo objeto. O **NMS** (*Non-Maximum Suppression*) remove caixas redundantes.

Ideia geral:

1. Ordenar as caixas por confiança.
2. Escolher a caixa de maior confiança.
3. Remover caixas muito sobrepostas com ela.
4. Repetir o processo.


In [ ]:
def nms(boxes, scores, iou_threshold=0.5):
    indices = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)
    keep = []

    while indices:
        current = indices.pop(0)
        keep.append(current)
        indices = [i for i in indices if iou(boxes[current], boxes[i]) < iou_threshold]

    return keep

boxes = [(50, 50, 150, 150), (60, 60, 155, 155), (200, 200, 300, 300)]
scores = [0.95, 0.85, 0.90]
print("Índices mantidos:", nms(boxes, scores, iou_threshold=0.5))


## Funcionamento Conceitual do YOLO Original

No YOLO original, a imagem era dividida em uma grade $(S 	\times S)$. Cada célula da grade era responsável por detectar objetos cujo centro caísse dentro dela.

Cada célula predizia:

- coordenadas de caixas delimitadoras;
- confiança das caixas;
- probabilidades de classes.

```text
Imagem 448x448
        ↓
Grade S x S
        ↓
Cada célula prediz B caixas + probabilidades de classes
        ↓
Filtragem por confiança + NMS
        ↓
Detecções finais
```


## 9. Evolução da Família YOLO

| Versão | Ideias principais |
|---|---|
| YOLOv1 | Detecção em tempo real com uma única rede neural. |
| YOLOv2 / YOLO9000 | Anchor boxes, batch normalization, treinamento em múltiplas escalas. |
| YOLOv3 | Detecção em múltiplas escalas, Darknet-53, melhorias incrementais. |
| YOLOv4 | Melhorias de treinamento e arquitetura, foco em velocidade e precisão. |
| YOLOv5 | Popularização via PyTorch e facilidade de uso. |
| YOLOv6 / YOLOv7 | Foco em aplicações industriais e alto desempenho. |
| YOLOv8 | Detecção, segmentação, classificação, pose e OBB; abordagem moderna e fácil de usar. |
| YOLO11 | Versão moderna da Ultralytics com suporte a múltiplas tarefas de visão computacional. |
| YOLO26 | Apontada na documentação da Ultralytics como a versão mais recente da família Ultralytics YOLO em 2026. |

Observação: depois do YOLOv3, diferentes grupos passaram a desenvolver versões com nomes YOLO. Por isso, nem todas as versões pertencem à mesma linha dos autores originais.

Mains informações sobre a familia YOLO:

- YOLO original: https://arxiv.org/abs/1506.02640
- Ultralytics YOLO11: https://docs.ultralytics.com/models/yolo11/
- Modelos suportados pela Ultralytics: https://docs.ultralytics.com/models/


## Estrutura Geral de Modelos YOLO Modernos

Apesar das diferenças entre versões, muitos modelos YOLO modernos podem ser entendidos por três blocos principais:

### Backbone

Responsável por extrair características da imagem: bordas, texturas, formas e partes de objetos.

### Neck

Combina características de diferentes escalas, importante para detectar objetos grandes e pequenos.

### Head

Produz as predições finais: caixas delimitadoras, classes e confiança.

```text
Imagem → Backbone → Neck → Head → Detecções
```


## Anchor-based versus anchor-free

Algumas versões do YOLO utilizam **âncoras** (*anchor boxes*).

Uma âncora é uma caixa pré-definida que serve como referência para a predição.

### Anchor-based

O modelo aprende ajustes em relação a caixas pré-definidas. Pode ajudar quando há padrões conhecidos de tamanho de objetos, mas exige escolha adequada das âncoras.

### Anchor-free

O modelo prediz objetos sem depender explicitamente de caixas âncora pré-definidas. Modelos modernos, como YOLOv8, são frequentemente descritos como usando uma abordagem **anchor-free**.


## Métricas Importantes em Detecção de Objetos

### Precision

$$
Precision = \frac{TP}{TP + FP}
$$

### Recall

$$
Recall =\frac{TP}{TP + FN}
$$

### AP — Average Precision

Área sob a curva precisão-revocação para uma classe.

### mAP — mean Average Precision

Média do AP entre várias classes.

Exemplos comuns:

- **mAP@0.5**: considera correta uma detecção com IoU ≥ 0.5;
- **mAP@0.5:0.95**: média do mAP para limiares de IoU de 0.5 até 0.95.


## Biblioteca Ultralytics

No Google Colab, execute:

```python
!pip install ultralytics
```

A biblioteca permite usar modelos YOLO para detecção de objetos, segmentação de instâncias, classificação, estimação de pose e detecção com caixas orientadas.


In [ ]:
# Execute esta célula no Google Colab se a biblioteca ainda não estiver instalada.
!pip install ultralytics


## Inferência com Modelo Pré-Treinado

Em ambientes sem acesso à internet, o download automático dos pesos pode falhar. No Google Colab, geralmente funciona normalmente.


In [ ]:
from ultralytics import YOLO

# Carrega um modelo pré-treinado pequeno.
# Dependendo da versão instalada da biblioteca, é possível utilizar:
# - yolo11n.pt
# - yolov8n.pt
# - yolo26n.pt, se disponível na sua instalação

model = YOLO("yolo11n.pt")

# Faz inferência em uma imagem de exemplo
results = model("https://ultralytics.com/images/bus.jpg")

for result in results:
    print(result.boxes)


## Visualizando o Resultado da Inferência


In [ ]:
import matplotlib.pyplot as plt

for result in results:
    annotated_image = result.plot()

    plt.figure(figsize=(10, 8))
    plt.imshow(annotated_image[:, :, ::-1])  # BGR para RGB
    plt.axis("off")
    plt.show()


## Caixas, Classes e Confianças

A saída do modelo contém coordenadas das caixas, classe prevista, confiança e nomes das classes.


In [ ]:
for result in results:
    names = result.names

    for box in result.boxes:
        cls_id = int(box.cls[0])
        conf = float(box.conf[0])
        xyxy = box.xyxy[0].tolist()

        print("Classe:", names[cls_id])
        print("Confiança:", round(conf, 3))
        print("Caixa [x1, y1, x2, y2]:", [round(v, 1) for v in xyxy])
        print("-" * 40)


## Detecção em Vídeo

Para usar YOLO em vídeo, basta passar o caminho de um arquivo `.mp4` ou uma câmera.

```python
results = model("video.mp4", save=True)
```

Para webcam local:

```python
results = model(0, show=True)
```

No Google Colab, o acesso direto à webcam exige procedimentos adicionais. Em um computador local, a opção `model(0)` costuma funcionar quando a câmera está disponível.


## Treinamento com Dataset Próprio

Estrutura típica:

```text
dataset/
├── images/
│   ├── train/
│   └── val/
├── labels/
│   ├── train/
│   └── val/
└── data.yaml
```

Cada imagem deve ter um arquivo `.txt` correspondente com as anotações.


## Formato das Anotações YOLO

Cada linha de um arquivo de anotação representa um objeto:

```text
classe x_centro y_centro largura altura
```

Os valores de posição e tamanho são **normalizados** entre 0 e 1.

Exemplo:

```text
0 0.52 0.43 0.18 0.25
```


In [ ]:
def xyxy_to_yolo(box, img_width, img_height):
    x1, y1, x2, y2 = box
    xc = ((x1 + x2) / 2) / img_width
    yc = ((y1 + y2) / 2) / img_height
    w = (x2 - x1) / img_width
    h = (y2 - y1) / img_height
    return xc, yc, w, h

img_width = 640
img_height = 480
box_pixels = (100, 80, 300, 220)
print(xyxy_to_yolo(box_pixels, img_width, img_height))


## Arquivo `data.yaml`

Exemplo:

```yaml
path: /content/dataset
train: images/train
val: images/val

names:
  0: pessoa
  1: carro
  2: bicicleta
```

Esse arquivo informa ao YOLO onde estão as imagens de treino, validação e os nomes das classes.


## Treinamento com Ultralytics


In [ ]:
from ultralytics import YOLO

model = YOLO("yolo11n.pt")

# Treina em um dataset próprio.
# Substitua pelo caminho correto do seu arquivo data.yaml.

# model.train(
#     data="/content/dataset/data.yaml",
#     epochs=50,
#     imgsz=640,
#     batch=16,
#     name="meu_experimento_yolo"
# )


## Hiperparâmetros Importantes

| Hiperparâmetro | Significado |
|---|---|
| `epochs` | Número de épocas de treinamento. |
| `imgsz` | Tamanho da imagem de entrada. |
| `batch` | Tamanho do lote. |
| `lr0` | Taxa de aprendizado inicial. |
| `patience` | Número de épocas sem melhora antes de parar. |
| `augment` | Uso de aumento de dados. |

Em aplicações reais, o ajuste desses parâmetros influencia fortemente o desempenho.


## Validação do Modelo

Após o treinamento, é possível avaliar o modelo no conjunto de validação.


In [ ]:
# Exemplo de validação
# metrics = model.val()
# print(metrics)


## Exportação para Implantação

Um modelo YOLO pode ser exportado para diferentes formatos, dependendo do ambiente de implantação.

```python
model.export(format="onnx")
model.export(format="torchscript")
model.export(format="tflite")
```

| Formato | Uso típico |
|---|---|
| PyTorch `.pt` | Uso direto em Python/PyTorch. |
| ONNX | Interoperabilidade e inferência otimizada. |
| TensorRT | Alta performance em GPUs NVIDIA. |
| TFLite | Dispositivos móveis e embarcados. |
| OpenVINO | Otimização para hardware Intel. |


In [ ]:
# Exemplo de exportação para ONNX
# model.export(format="onnx")


## Pipeline Completo de um Projeto com YOLO

```text
1. Definir o problema
2. Coletar imagens
3. Anotar objetos nas imagens
4. Separar treino, validação e teste
5. Treinar modelo YOLO
6. Avaliar mAP, precision e recall
7. Testar em imagens e vídeos reais
8. Exportar modelo
9. Implantar no dispositivo final
10. Monitorar desempenho em campo
```


## Boas Práticas

1. **Use dados representativos:** o modelo precisa ver no treinamento situações parecidas com as do uso real.
2. **Cuide da anotação:** caixas mal anotadas prejudicam o aprendizado.
3. **Separe corretamente treino e validação:** evite vazamento de dados.
4. **Avalie em cenários reais:** métricas em dataset podem não representar o desempenho em campo.
5. **Considere custo computacional:** modelos maiores tendem a ser mais precisos, mas podem ser lentos para sistemas embarcados.
6. **Analise erros:** veja falsos positivos, falsos negativos e confusões entre classes.


# Referências

* Redmon, J.; Divvala, S.; Girshick, R.; Farhadi, A. **You Only Look Once: Unified, Real-Time Object Detection**. CVPR, 2016. https://arxiv.org/abs/1506.02640

* Redmon, J.; Farhadi, A. **YOLO9000: Better, Faster, Stronger**. 2016. https://arxiv.org/abs/1612.08242

* Redmon, J.; Farhadi, A. **YOLOv3: An Incremental Improvement**. 2018. https://arxiv.org/abs/1804.02767

* Bochkovskiy, A.; Wang, C.-Y.; Liao, H.-Y. M. **YOLOv4: Optimal Speed and Accuracy of Object Detection**. 2020. https://arxiv.org/abs/2004.10934

* Ultralytics YOLO Documentation. https://docs.ultralytics.com/

* Ultralytics YOLO11 Documentation. https://docs.ultralytics.com/models/yolo11/

* Ultralytics Models Documentation. https://docs.ultralytics.com/models/
